# 03b — PVI calculation

A Cook-style PVI for every district on every set of lines, built from `partisan_lean.csv` (district results by map) and
`presidential.csv` (national results), one row per `(year, state, district)`:

`year, state, district, cook_pvi, partisan_lean, sources, quality_flag`

For lines first used in House election year `year`, `prior` is the latest presidential election **before** `year` and
`anteprior` the one before that. The election held in `year` itself is never used, so the 2008 lines use 2004 and 2000.

* `rel(p) = dem_pct_2p_norm(district, p) − dem_pct_2p_norm(national, p)`, the district's two-party Democratic share relative to the nation
* `cook_pvi = 0.75 · rel(prior) + 0.25 · rel(anteprior)`, positive = more Democratic than the nation
* `partisan_lean = 2 · cook_pvi`, the same lean expressed as a margin
* `sources`: the two rows' distinct sources, `"; "`-joined, prior first
* `quality_flag`: the worse of the two rows' flags, ordered `two_party` < `approximate` < `backsolved` < `cook_rounded`

Lines missing either election entirely would be dropped. The two that lack one in the raw data are filled in `03a` by backsolving
Cook's published PVIs: 2016 on the 2022 lines (Cook 2023, 75/25, so their PVI reproduces Cook's) and 2004 on the 2012 lines
(Cook's 2012 PVI, 50/50 and rounded to whole numbers).

The 2026 lines have no results of their own; see the 2026 section below.

In [1]:
import json
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../../data/raw/pvi.sources")
INTERIM_DIR = Path("../../data/interim")
HOUSE_INTERIM_DIR = INTERIM_DIR / "house"  # House-specific (district-level) interim data
REFERENCE_DIR = Path("../../data/reference")

COLUMNS = ["year", "state", "district", "cook_pvi", "partisan_lean", "sources", "quality_flag"]
WEIGHTS = (0.75, 0.25)  # (prior, anteprior) presidential election
QUALITY_ORDER = ["two_party", "approximate", "backsolved", "cook_rounded"]  # best -> worst

lean = pd.read_csv(HOUSE_INTERIM_DIR / "partisan_lean.csv")
national = pd.read_csv(INTERIM_DIR / "presidential.csv")

In [2]:
def prior_pres_years(year: int, pres_years: pd.Series) -> tuple[int, int]:
    """The two latest presidential years strictly before `year`, latest first."""
    earlier = sorted(p for p in pres_years if p < year)
    if len(earlier) < 2:
        raise ValueError(f"Fewer than two presidential elections before {year}")
    return earlier[-1], earlier[-2]


def build_pvi(lean: pd.DataFrame, national: pd.DataFrame) -> pd.DataFrame:
    """Weight each district's two-party lean relative to the nation over the two presidential elections before its map year."""
    unknown = set(lean["quality_flag"]) - set(QUALITY_ORDER)
    if unknown:
        raise ValueError(f"Unranked quality flags: {unknown}")
    rel = lean.merge(
        national[["pres_year", "dem_pct_2p_norm"]], on="pres_year", suffixes=("", "_national"), validate="many_to_one"
    )
    rel["rel"] = rel["dem_pct_2p_norm"] - rel["dem_pct_2p_norm_national"]

    frames = []
    for year in sorted(lean["dist_map_year"].unique()):
        prior, anteprior = prior_pres_years(year, national["pres_year"])
        on_map = rel[rel["dist_map_year"] == year]
        absent = [p for p in (prior, anteprior) if p not in set(on_map["pres_year"])]
        if absent:
            print(f"Dropping {year}: needs {prior} and {anteprior}, but the {year} lines have no {absent} results")
            continue
        both = on_map[on_map["pres_year"].isin([prior, anteprior])].pivot(
            index=["state", "district"], columns="pres_year", values=["rel", "source", "quality_flag"]
        )
        if both.isna().any().any():
            partial = both[both.isna().any(axis=1)].index.tolist()
            raise ValueError(f"{year}: districts missing {prior} or {anteprior}: {partial}")
        frames.append(pd.DataFrame({
            "year": year,
            "cook_pvi": WEIGHTS[0] * both[("rel", prior)] + WEIGHTS[1] * both[("rel", anteprior)],
            "sources": ["; ".join(dict.fromkeys(pair)) for pair in zip(both[("source", prior)], both[("source", anteprior)])],
            "quality_flag": [
                max(pair, key=QUALITY_ORDER.index)
                for pair in zip(both[("quality_flag", prior)], both[("quality_flag", anteprior)])
            ],
        }).reset_index())
    pvi = pd.concat(frames, ignore_index=True)
    pvi["partisan_lean"] = 2 * pvi["cook_pvi"]
    return pvi[COLUMNS].sort_values(["year", "state", "district"], ignore_index=True)


pvi = build_pvi(lean, national)
pvi

,year,state,district,cook_pvi,partisan_lean,sources,quality_flag
0,2006,AK,0,-13.252993,-26.505985,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
1,2006,AL,1,-12.924992,-25.849984,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
2,2006,AL,2,-14.788062,-29.576123,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
3,2006,AL,3,-6.204728,-12.409457,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
4,2006,AL,4,-18.483124,-36.966249,https://docs.google.com/spreadsheets/d/1l7W130...,approximate
...,...,...,...,...,...,...,...
4345,2024,WI,7,-12.26665,-24.533299,https://christophertkenny.com/atlas,two_party
4346,2024,WI,8,-10.165608,-20.331216,https://christophertkenny.com/atlas,two_party
4347,2024,WV,1,-23.231234,-46.462468,https://christophertkenny.com/atlas,two_party
4348,2024,WV,2,-21.523224,-43.046449,https://christophertkenny.com/atlas,two_party


## 2026 lines

No presidential results exist on the 2026 lines, and Cook's 2026 PVI can't be backsolved: each district gives one equation
(`0.75·rel(2024) + 0.25·rel(2020)`) in two unknowns. But most states didn't redraw for 2026:

* **Unchanged states**: their 2026 lines are their 2024 lines, so the 2024-lines results for 2024 and 2020 are carried over and
  the PVI is computed exactly as above.
* **Redrawn states**: Cook's 2026 PVI is used directly. It is published rounded to whole numbers (±0.5), so `quality_flag = "cook_rounded"`.

A state counts as redrawn if any of its districts' PVIs differ between Cook's 2025 release (2024 lines) and its 2026 release.

In [3]:
FILE_COOK_2026 = "cook_pvi_public_2026.csv"
SOURCE_COOK = "https://www.cookpolitical.com/cook-pvi"
QUALITY_COOK_ROUNDED = "cook_rounded"


def parse_rounded_pvi(pvi: pd.Series) -> pd.Series:
    """Parse whole-number PVIs (`D+x` / `R+x` / `EVEN`) to signed numbers, D positive."""
    parts = pvi.replace("EVEN", "D+0").str.extract(r"^([DR])\+(\d+)$")
    if parts.isna().any().any():
        raise ValueError(f"Unexpected PVI values: {pvi[parts.isna().any(axis=1)].tolist()}")
    return parts[0].map({"D": 1, "R": -1}) * parts[1].astype(int)


def clean_cook_2026(path: Path) -> pd.DataFrame:
    """Read Cook's 2025/2026 PVI list, split `Dist` into state and district, and parse both PVIs."""
    # the header names five columns, but every row carries a sixth (the 2026 rank)
    cook = pd.read_csv(
        path, header=0, names=["Dist", "Incumbent", "Party", "2025 PVI", "2026 PVI", "Rank"], dtype=str, index_col=False
    )
    dist = cook["Dist"].str.extract(r"^([A-Z]{2})-(AL|\d{2})$")
    if dist.isna().any().any():
        raise ValueError(f"Unexpected Dist values: {cook.loc[dist.isna().any(axis=1), 'Dist'].tolist()}")
    return cook.assign(
        state=dist[0],
        district=dist[1].replace("AL", "0").astype(int),
        **{"2025 PVI": parse_rounded_pvi(cook["2025 PVI"]), "2026 PVI": parse_rounded_pvi(cook["2026 PVI"])},
        Rank=cook["Rank"].astype(int),
    )[["state", "district", "Incumbent", "Party", "2025 PVI", "2026 PVI", "Rank"]]


cook_2026 = clean_cook_2026(RAW_DIR / FILE_COOK_2026)
cook_2026.to_csv(HOUSE_INTERIM_DIR / "cook_pvi_public_2026_cleaned.csv", index=False)

redrawn = sorted(cook_2026.loc[cook_2026["2025 PVI"] != cook_2026["2026 PVI"], "state"].unique())
print(f"Redrawn for 2026 ({len(redrawn)} states): {redrawn}")

carried = lean[(lean["dist_map_year"] == 2024) & ~lean["state"].isin(redrawn)].assign(dist_map_year=2026)
pvi_2026_unchanged = build_pvi(carried, national)
redrawn_rows = cook_2026[cook_2026["state"].isin(redrawn)]
pvi_2026_redrawn = pd.DataFrame({
    "year": 2026,
    "state": redrawn_rows["state"],
    "district": redrawn_rows["district"],
    "cook_pvi": redrawn_rows["2026 PVI"].astype(float),
    "partisan_lean": 2 * redrawn_rows["2026 PVI"].astype(float),
    "sources": SOURCE_COOK,
    "quality_flag": QUALITY_COOK_ROUNDED,
})
pvi = pd.concat([pvi, pvi_2026_unchanged, pvi_2026_redrawn])[COLUMNS].sort_values(["year", "state", "district"], ignore_index=True)
pvi.query("year == 2026").groupby("quality_flag").size()

Redrawn for 2026 (9 states): ['AL', 'CA', 'FL', 'LA', 'NC', 'OH', 'TN', 'TX', 'UT']


quality_flag
cook_rounded    173
two_party       262
dtype: int64

In [4]:
# our unrounded PVI in the unchanged states should agree with Cook's rounded 2026 PVI to within rounding (plus source differences)
check_2026 = pvi_2026_unchanged.merge(cook_2026[["state", "district", "2026 PVI"]], on=["state", "district"], validate="one_to_one")
check_2026["diff"] = check_2026["cook_pvi"] - check_2026["2026 PVI"]
print(f"{len(check_2026)} unchanged districts: mean |ours − Cook| {check_2026['diff'].abs().mean():.2f}, "
      f"{(check_2026['diff'].abs() <= 0.5).mean():.0%} within rounding")
check_2026[check_2026["diff"].abs() > 1]

262 unchanged districts: mean |ours − Cook| 0.26, 93% within rounding


,year,state,district,cook_pvi,partisan_lean,sources,quality_flag,2026 PVI,diff
153,2026,NJ,8,16.599144,33.198287,https://christophertkenny.com/atlas,two_party,15,1.599144


## Sanity checks

In [5]:
def check_districts(df: pd.DataFrame, path: Path = REFERENCE_DIR / "district_counts.json") -> None:
    """Check every year has exactly the expected districts per state: [0] for at-large, else 1..n."""
    with open(path) as f:
        counts = json.load(f)
    expected = {
        (year, state): [0] if n == 1 else list(range(1, n + 1))
        for year in df["year"].unique() for state, n in counts[str(year)].items()
    }
    found = df.groupby(["year", "state"])["district"].agg(sorted).to_dict()
    if found != expected:
        bad = sorted(k for k in found.keys() | expected.keys() if found.get(k) != expected.get(k))
        raise ValueError(f"Districts don't match district_counts.json for (year, state): {bad}")


check_districts(pvi)
if pvi[["cook_pvi", "partisan_lean"]].isna().any().any():
    raise ValueError("Missing cook_pvi / partisan_lean")
if not (pvi["partisan_lean"] == 2 * pvi["cook_pvi"]).all():
    raise ValueError("partisan_lean != 2 * cook_pvi")
pvi.groupby("year").agg(
    districts=("district", "size"),
    mean_pvi=("cook_pvi", "mean"),
    min_pvi=("cook_pvi", "min"),
    max_pvi=("cook_pvi", "max"),
    quality_flag=("quality_flag", "unique"),
)

,districts,mean_pvi,min_pvi,max_pvi,quality_flag
year,,,,,
2006,435,0.725896,-27.609498,42.186095,[approximate]
2008,435,0.725896,-27.609498,42.186095,[approximate]
2010,435,0.745774,-29.705639,41.269109,[approximate]
2012,435,0.067514,-30.233313,41.155576,[backsolved]
2014,435,0.305876,-32.781802,44.098121,[approximate]
2016,435,0.285114,-32.781802,44.098121,[approximate]
2018,435,0.484817,-33.546007,44.198407,[approximate]
2020,435,0.475713,-33.546007,44.198407,[approximate]
2022,435,0.398779,-33.24,39.7,"[backsolved, approximate]"


At-large states are whole states, so their PVI can be compared directly with Cook's published statewide PVIs for the
release using the same two elections (e.g. the 2024 lines here use 2016/2020, like Cook's 2022 release).

In [6]:
pvi.query("district == 0").pivot(index="state", columns="year", values="cook_pvi").round(1)

year,2006,2008,2010,2012,2014,2016,2018,2020,2022,2024,2026
state,,,,,,,,,,,
AK,-13.252993,-13.252993,-13.795845,-13.865896,-10.635119,-10.635119,-9.457667,-9.457667,-8.02,-8.023496,-6.494408
DE,5.192752,5.192752,7.897896,7.981487,7.841883,7.841883,5.557264,5.557264,6.74,6.739178,7.966575
MT,-10.221148,-10.221148,-5.787909,-5.912606,-7.970137,-7.970137,-11.449968,-11.449968,NaN,NaN,NaN
ND,-13.084698,-13.084698,-8.925954,-9.066079,-11.113518,-11.113518,-18.731074,-18.731074,-19.85,-19.837336,-18.328729
SD,-10.358511,-10.358511,-8.322986,-8.515006,-10.38224,-10.38224,-15.616889,-15.616889,-16.05,-16.047424,-14.60459
VT,9.877736,9.877736,14.110533,14.106796,16.001581,16.001581,14.610405,14.610405,15.54,15.542943,16.859406
WY,-19.723649,-19.723649,-19.802578,-20.143709,-22.437235,-22.437235,-25.900744,-25.900744,-25.27,-25.267226,-23.23438


In [7]:
pvi.to_csv(HOUSE_INTERIM_DIR / "house_partisan_lean.csv", index=False)